## <span style="color: #eab308">Table of Contents</span>

- [Current Project Status](#current-project-status)
- [Tech Stack and Architecture](#tech-stack-and-architecture)
- [Environment Variables You Need](#environment-variables-you-need)
- [How the App Starts, Step by Step](#how-the-app-starts-step-by-step)
- [What to Type in the Terminal](#what-to-type-in-the-terminal)
- [Troubleshooting and Notes](#troubleshooting-and-notes)
- [Question set 1](#question-set-1)
- [Question set 2](#question-set-2)
- [New Plan](#new-plan)
- [AI-SDK BLOCKING](#ai-sdk-blocking)
- [AI_SDK STREAMING](#ai-sdk-streaming)
- [INNGEST](#inngest)
- [INNGEST Diagram](#inngest-diagram)
- [Start-All_Terminals](#start-all-terminals)

> **Q: what is page.tsx is it a vite project?**
>
> No — this is a **Next.js** project (App Router), not Vite. There's no `vite.config.ts`, and `package.json`'s scripts use `next dev`/`next build`/`next start`, confirmed by `next.config.ts` and `next` v16.1.1 in dependencies.
>
> **What `page.tsx` is:** In Next.js's App Router, `src/app/page.tsx` is a special filename the framework recognizes automatically — it defines the UI for a route based on its folder location. Since it lives directly in `src/app/`, it's the component rendered for the root route `/` (the homepage). If you later added `src/app/settings/page.tsx`, that would automatically become the page for `/settings` — no manual route configuration needed, unlike Vite (which is just a bundler/dev-server and leaves routing entirely up to you, e.g. via React Router).
>
> So the rough equivalence is:
> - **Vite** = a build tool/dev server, routing is DIY.
> - **Next.js** = a full framework where the file system *is* the router, and `page.tsx` is the reserved name for "this folder's route content." `layout.tsx` (which this project also has) is the matching reserved name for shared wrapper UI around a page.

> **Q: what is the role of page.tsx and layout.tsx?**
>
> In Next.js's App Router, these are two reserved filenames with distinct jobs:
>
> **`src/app/layout.tsx` — the shared shell.** Wraps every page under it and stays mounted across navigations (it doesn't re-render when you move between routes). In this project it's the *root* layout, so it wraps the entire app. It sets up `<html>`/`<body>`, fonts, and global metadata; wraps everything in `ClerkProvider` → `ThemeProvider` → `ConvexClientProvider` so auth, theme, and the Convex connection are available to every page; renders the header (Sign In/Sign Up buttons or `UserButton`) that appears on every route; and renders `{children}` — that's where the current page's content gets slotted in.
>
> **`src/app/page.tsx` — the actual route content.** Defines what's unique to one specific route. Since it sits directly in `src/app/`, it's the page for `/` — this project's homepage. It's *inside* `{children}` from the layout above it, and holds the stuff that's specific to this one screen: the "Create Project" form, the live projects list, the mode toggle — none of which appear on other routes, unlike the header in `layout.tsx`.
>
> **The relationship:** for any URL, Next.js renders every `layout.tsx` from the root down to that route, nested outside-in, with that route's `page.tsx` as the innermost `{children}`. Right now there's only one layout and one page, so it's simple: `layout.tsx` → `page.tsx`. If you added `src/app/settings/page.tsx`, it would automatically inherit this same root layout (header, providers, fonts) and only need to define its own unique content — that's the reuse layouts are for.

> **Q: what is the idea of wrapping Clerk->Theme->convex Is this a react thing or nextjs?**
>
> It's a **React pattern**, not Next.js-specific — Next.js just gives you `layout.tsx` as a canonical place to do it once, app-wide.
>
> **The underlying mechanism: React Context.** Each of `ClerkProvider`, `ThemeProvider`, and `ConvexClientProvider` (internally, `ConvexProviderWithClerk`) is a Context Provider — a component that makes some value (session state, theme, a Convex client) available to every descendant via `useContext`-based hooks (`useAuth`, `useTheme`, `useConvexAuth`, etc.), without threading props down manually through every level. This is plain React, and works identically in a Vite app, Create React App, or anywhere else.
>
> **Why this specific nesting order matters — and it's not arbitrary:**
> ```
> ClerkProvider
>   -> ThemeProvider
>     -> ConvexClientProvider   (internally: ConvexProviderWithClerk)
> ```
> `ConvexProviderWithClerk` (`src/app/ConvexClientProvider.tsx`) calls Clerk's `useAuth()` internally to read the session and attach a token to the Convex connection. `useAuth()` only works if a `ClerkProvider` exists somewhere *above* it in the tree — React context lookups walk upward, so a child can read a provider's value, but a provider can't read one nested inside it. So **Clerk must wrap Convex, not the other way around** — that's a hard dependency, not a style choice. `ThemeProvider` in the middle has no such dependency on Clerk or Convex (it only manages a CSS class via `next-themes`), so its position between the two is really just convention/readability here, not a requirement.
>
> **Where Next.js comes in:** it doesn't invent this pattern — it just gives you exactly one obvious place to set it up once for the whole app: the root `src/app/layout.tsx`, since every route renders inside it. In a plain Vite/React app you'd do the identical nesting by hand in your top-level `App.tsx` or `main.tsx` instead.

<a id="current-project-status"></a>

## <span style="color: #eab308">Current Project Status</span>

**Project:** `my_polaris`.

**What's actually built, functionally:**
- A single homepage (`src/app/page.tsx`) that renders:
  - A header with Clerk **Sign In** / **Sign Up** buttons when signed out, and a **UserButton** (avatar menu) when signed in — defined in `src/app/layout.tsx`.
  - A light/dark **mode toggle** (`src/components/mode-toggle.tsx`, powered by `next-themes`).
  - A **"Project name" input + "Create Project" button**, wired to a real Convex mutation.
  - A **live list** of the signed-in user's projects, wired to a real Convex query — updates automatically, no refresh needed.
- A Convex backend (`convex/`) with one table, `projects` (fields: `name`, `ownerId`, `importStatus`), and two functions in `convex/projects.ts`: `create1` (insert a project) and `get1` (list the current user's projects).
- Convex is wired to Clerk for authentication (`convex/auth.config.ts` + `ConvexProviderWithClerk` in `src/app/ConvexClientProvider.tsx`), so every Convex call knows *who* is calling and can be scoped to that user.
- Around 60 shadcn/ui components are already generated under `src/components/ui/` (button, input, dialog, sidebar, calendar, chart, form, etc.). Most are **not wired into the page yet** — they're scaffolding the `shadcn` CLI generated for future use.

**In short:** this is an early-stage scaffold — working auth, a working database, and one working create/list flow — not yet a full multi-page app.

<a id="tech-stack-and-architecture"></a>

## <span style="color: #eab308">Tech Stack and Architecture</span>

> **Core stack**
> | Layer | Tool | Role |
> |---|---|---|
> | Framework | Next.js 16.1.1 (App Router, Turbopack) | Routing, rendering, dev server |
> | UI runtime | React 19.2.3 / react-dom 19.2.3 | Component rendering |
> | Styling | Tailwind CSS v4 (+ `tailwind-merge`, `tw-animate-css`) | Utility-class styling |
> | Component library | shadcn/ui on Radix primitives (`radix-ui`, `@base-ui/react`) | Pre-built accessible UI components in `src/components/ui/` |
> | Auth | Clerk (`@clerk/nextjs`) | Sign in/up, sessions, `src/proxy.ts` middleware |
> | Backend/data | Convex (`convex`) | Realtime database + serverless functions, synced via `npx convex dev` |
> | Auth <-> Convex bridge | `convex/react-clerk`'s `ConvexProviderWithClerk` | Attaches the Clerk session token to every Convex request |
> | Forms (installed, not yet used) | `react-hook-form` + `zod` + `@hookform/resolvers` | Form state + validation, ready for future forms |
> | Icons | `lucide-react` | Icon set used by shadcn components |
> | Theming | `next-themes` | System/light/dark mode |
> | Language | TypeScript 5 | Type safety across app + Convex functions |

**Request/render flow:**

Browser → Next.js dev server → `src/app/layout.tsx` renders providers in this order:

```
ClerkProvider                 (Clerk session available everywhere)
  -> ThemeProvider             (next-themes: applies light/dark class)
    -> ConvexClientProvider    (opens the Convex WebSocket, attaches Clerk token)
      -> page content (src/app/page.tsx)
```

The Convex client talks **directly** to your Convex cloud deployment over its own WebSocket — it is not proxied through the Next.js server. `src/proxy.ts` (Clerk's middleware, renamed from `middleware.ts` in Next.js 16) runs server-side on every matching request to attach auth context, but it does not block any routes in this project yet.

<a id="environment-variables-you-need"></a>

## <span style="color: #eab308">Environment Variables You Need</span>

Two files hold them, and neither is meant to be shared:

- **`.env`** — Clerk keys + Convex keys. Loaded by both the Next.js server and, where prefixed `NEXT_PUBLIC_`, the browser.
- **`.env.local`** — duplicates the three Convex vars (a leftover from before they were moved into `.env`; harmless as long as the values agree, since Next.js merges both files).

> **Variables (names only — values are secrets, not shown)**
> | Variable | Used in | Purpose |
> |---|---|---|
> | `NEXT_PUBLIC_CLERK_PUBLISHABLE_KEY` | `ClerkProvider` (`src/app/layout.tsx`) | Clerk's public/browser-safe key |
> | `CLERK_SECRET_KEY` | `clerkMiddleware()` (`src/proxy.ts`) | Clerk's server-side secret |
> | `CLERK_JWT_ISSUER_DOMAIN` | `convex/auth.config.ts` | Lets Convex verify Clerk's session JWTs |
> | `CONVEX_DEPLOYMENT` | read by the `convex` CLI | Tells `npx convex dev` / `npx convex deploy` which deployment to sync |
> | `NEXT_PUBLIC_CONVEX_URL` | `ConvexClientProvider.tsx` | The browser-facing Convex deployment URL |
> | `NEXT_PUBLIC_CONVEX_SITE_URL` | generated by `convex dev` | Convex's HTTP Actions site URL (not used by any code yet) |

If you ever copy this project to a new machine or hand it to someone else, they will need their own copies of these two files — that's expected; these files simply live on disk, local to each machine.

<a id="how-the-app-starts-step-by-step"></a>

## <span style="color: #eab308">How the App Starts, Step by Step</span>

**1. `npx convex dev` connects to your Convex deployment**
Reads `CONVEX_DEPLOYMENT`, pushes `convex/schema.ts` and every function in `convex/*.ts` to that deployment, and — while left running — keeps `convex/_generated/` (`api.d.ts`, `api.js`, `server.d.ts`, …) in sync. This generated folder is what makes `api.projects.create1` and `api.projects.get1` exist as typed references in `src/app/page.tsx`. Nothing Convex-related works without this running.

**2. `npm run dev` starts the Next.js dev server**
Turbopack builds and serves the app on `http://localhost:3000`.

**3. Browser requests `/` → `src/app/layout.tsx` renders**
`ClerkProvider` wraps everything first, reading `NEXT_PUBLIC_CLERK_PUBLISHABLE_KEY` so Clerk's session state (signed in / signed out) is available app-wide.

**4. `ThemeProvider` (next-themes) applies the theme class**
Before paint, so the correct light/dark styling shows immediately (`suppressHydrationWarning` on `<html>` avoids a server/client mismatch warning here).

**5. `ConvexClientProvider` opens the Convex connection**
Creates one `ConvexReactClient` pointed at `NEXT_PUBLIC_CONVEX_URL`, wrapped in `ConvexProviderWithClerk`, which calls Clerk's `useAuth` to fetch the current session token and attaches it to the Convex WebSocket connection.

**6. `src/proxy.ts`'s `clerkMiddleware()` runs server-side**
On every matching request (per its `config.matcher`), attaching auth context so server components / route handlers *could* call `auth()` or `currentUser()` — nothing in this project does yet, but the wiring is in place.

**7. `Home` (`src/app/page.tsx`) mounts in the browser**
`useConvexAuth()` reports whether Clerk **and** Convex both agree you're signed in. While `isAuthenticated` is `false`, the `get1` query is skipped (passed `"skip"` instead of `{}`). Once `true`, it subscribes for real, and your projects list appears — updating live on every future create, with no manual refresh.

<a id="what-to-type-in-the-terminal"></a>

## <span style="color: #eab308">What to Type in the Terminal</span>

You need **two terminals running at the same time** — one for Convex, one for Next.js. Both stay open while you work; close either with `Ctrl+C` when you're done for the day.

**Terminal A — Convex (leave running):**
```bash
cd my_polaris
npx convex dev
```
First time ever on a machine, this will prompt you to log in to Convex and confirm the project link; after that it just syncs silently in the background.

**Terminal B — Next.js (leave running):**
```bash
cd my_polaris
npm run dev
```

Then open **http://localhost:3000** in your browser.

**If `node_modules` is missing, or after pulling changes that touch `package.json`:**
```bash
npm install
```
Run this once before `npm run dev`.

> **Quick reference**
> | Command | What it does |
> |---|---|
> | `npm run dev` | Start the Next.js dev server (Turbopack, hot reload) |
> | `npx convex dev` | Sync `convex/` functions + schema, watch for changes |
> | `npm run build` | Production build |
> | `npm run start` | Run the production build locally |
> | `npm run lint` | Run ESLint |

<a id="troubleshooting-and-notes"></a>

## <span style="color: #eab308">Troubleshooting and Notes</span>

**Sign In / Sign Up buttons don't render, or a Clerk error appears**
Check `.env`'s `NEXT_PUBLIC_CLERK_PUBLISHABLE_KEY` and `CLERK_SECRET_KEY` are set, and restart `npm run dev` — environment variables are only read when the dev server starts, so editing `.env` while it's running has no effect until restart.

**"Create Project" throws `Not authenticated`**
You're not signed in. `ctx.auth.getUserIdentity()` in `convex/projects.ts` requires an active Clerk session — sign in first via the header button.

**Convex calls fail entirely, or `api.projects...` shows as undefined in the editor**
Make sure `npx convex dev` is actually running in its own terminal — it's what keeps `convex/_generated/` and the live schema in sync. The Next.js dev server does **not** start it for you; the two are independent processes.

**`npm error ENOENT ... Could not read package.json`**
This happened once before, in an earlier session, because `npm` was run from the *parent* folder (the old `polaris/`, one level above where `package.json` actually lives) instead of from inside `my_polaris/`. Always run `npm`/`npx` commands from inside the folder that directly contains `package.json`.

### One subtlety
`.env` and `.env.local` both define the same three Convex variables. Next.js loads `.env.local` with higher precedence than `.env`, so `.env.local`'s copies are the ones actually in effect — the `.env` copies are redundant but harmless as long as both stay in sync. If they ever drift apart, `.env.local` wins.

<a id="question-set-1"></a>

## <span style="color: #eab308">Question set 1</span>

> **Q: explain the Input and its handlers. When Enter is pressed, then handleCreate then a project is created in convex. so what is onChange doing?**
>
> Line by line:
> - **`<Input placeholder="Project name" ...>`** — the actual text field.
> - **`value={newProjectName}`** — this makes it a **controlled component**: whatever the box displays is dictated entirely by React state (`newProjectName`, from `useState("")`), not by the browser's own internal input state.
> - **`onChange={(e) => setNewProjectName(e.target.value)}`** — fires on **every single keystroke** (each character typed or deleted), not on Enter.
> - **`onKeyDown={(e) => { if (e.key === "Enter") handleCreate(); }}`** — fires on every key press too, but only *acts* when that key is `"Enter"`.
>
> **What `onChange` is doing, specifically:** it has nothing to do with Convex or creating the project — it's purely local UI bookkeeping. Because `value={newProjectName}` pins the box's displayed text to React state, the box can *only* show new characters if something updates that state in response to typing. `onChange` is that something: each keystroke calls `setNewProjectName(e.target.value)`, which re-renders the component with the box's `value` now equal to whatever you just typed.
>
> If you deleted `onChange`, the input would appear **frozen** — you could click into it and type, but since `value` never changes (nothing updates `newProjectName`), React would keep snapping the displayed text back to the old value on every render. This is the classic "controlled input needs `value` + `onChange` together" pairing.
>
> **So the full sequence when you type "My App" and hit Enter:**
> 1. Each keystroke (`M`, `y`, ` `, `A`, `p`, `p`) individually triggers `onChange` → `newProjectName` accumulates to `"My App"` in state, purely client-side, no network calls.
> 2. Pressing Enter triggers `onKeyDown` → `handleCreate()` runs, which reads the *already-typed* `newProjectName` from state, trims it, and sends it to Convex via `createProject({ name })`.
>
> So `onChange` is what lets you see what you're typing at all; `onKeyDown`/`handleCreate` is what happens *after*, using the value `onChange` already built up.

> **Q: after I press Enter, handleCreate is creating a new project in convex, but how the name that was typed is used by handleCreate?**
>
> Via **closure**, not an argument.
> ```tsx
> const [newProjectName, setNewProjectName] = useState("");
>
> const handleCreate = async () => {
>   const name = newProjectName.trim();
>   ...
>   await createProject({ name });
> };
> ```
> `handleCreate` takes no parameters. `Home` is a function component, so it re-runs top-to-bottom on every state change, and a **new** `handleCreate` function is created each render — that new function closes over (captures) whatever `newProjectName` equals *in that render*. Every keystroke's `onChange` (`setNewProjectName(e.target.value)`) triggers a re-render, producing a fresh `handleCreate` bound to the just-typed value, which gets wired up again to the `<Input>`'s `onKeyDown` and the `<Button>`'s `onClick`. So when you press Enter, the `handleCreate` that actually runs is whichever one is currently attached — the one from the most recent render — and it reads `newProjectName` straight out of that closure, not as an argument passed in.

> **Q: why is it `handleCreate` (no parens) on the button's `onClick`, but `handleCreate()` (with parens) inside the `onKeyDown` handler? If the `onKeyDown` handler already creates the project, what does the button do?**
>
> **Summary:** both trigger the exact same `handleCreate`, with the exact same result — pressing Enter and clicking the button are genuinely equivalent. The parens difference is just the two forms described below: the button passes a bare reference (`{handleCreate}`), while `onKeyDown` has to use an inline wrapper (`{(e) => { if (e.key === "Enter") handleCreate(); }}`) because it needs to filter out every non-Enter key before deciding to call it — a bare reference there would fire on every single keystroke instead.
>
> Given both paths lead to the same place, the button still earns its spot for three reasons: **discoverability** (not everyone expects Enter to submit), **touch devices** (no reliable Enter-equivalent there), and **visible feedback** — the button visibly grays out via `disabled={!newProjectName.trim()}` when the input is empty, while the Enter-key path just silently does nothing in that case (`handleCreate`'s own guard exits quietly, with no visual cue).
>
> As for how a bare reference like `{handleCreate}` ever actually gets called, and why render timing isn't what triggers it — see the syntax breakdown below, which explains the general mechanism (bare reference vs. inline wrapper) that this specific case is just one example of.

> **Q: my question is about how each handler is wired up in the code — is it a syntax issue or a pattern issue? What is the syntax of `onClick={handleCreate}` compared to the `onKeyDown` handler? Could I write `onClick` as an arrow function that invokes `handleCreate()` immediately instead?**
>
> It's a **pattern choice, not a syntax requirement.** JSX doesn't have two different kinds of event-prop syntax; `onClick` and `onKeyDown` both follow the exact same rule.
>
> **The one syntax rule:** any JSX attribute is written `propName={expression}` — the `{}` just embeds a JavaScript expression, and whatever that expression evaluates to becomes the prop's value. `value={newProjectName}` and `onClick={handleCreate}` follow the identical rule — one just happens to hold a string, the other happens to hold a function.
>
> Given that one rule, there are two kinds of expression you can put inside `{}`, and both are equally legal on `onClick` **or** `onKeyDown`:
> 1. **A bare identifier** — `{handleCreate}` — evaluates to the function object itself. React stores that reference and calls it later, passing the event as the argument.
> 2. **An inline arrow function** — `{() => handleCreate()}` or `{(e) => { ... }}` — evaluates to a *new* function, created fresh each render, whose body you fully control. React calls *this* one on the event, and your code decides what happens next.
>
> **So yes — this would work, and behave identically to what's already there:**
> ```tsx
> <Button onClick={() => handleCreate()} disabled={!newProjectName.trim()}>
> ```
> The wrapper receives the click event as an implicit argument, ignores it, and calls `handleCreate()` explicitly. Same outcome either way.
>
> **So why is one written bare and the other wrapped, if both forms work everywhere?** Purely because of what each one *needs* to do before deciding to call `handleCreate`:
> - The button's click needs **no filtering** — a click is always meant to trigger creation. The bare reference is the simplest sufficient form; wrapping it would work but is unnecessary extra code.
> - `onKeyDown` fires on *every* key, so it **must** run some logic first (`if (e.key === "Enter")`) to decide *whether* to call `handleCreate` at all. That logic has to live somewhere, so an inline arrow function is the only way to fit it in.
>
> **One related gotcha worth knowing:** the bare-reference form only works safely when the function either takes no parameters (like `handleCreate`) or expects the event as its first argument — because React always calls it with the event. If a handler needed some *other* value — say, a function like `handleDelete(projectId)` — you'd be forced into the wrapper form: `onClick={() => handleDelete(project._id)}`, never `onClick={handleDelete}`, because the bare form would incorrectly hand it the click event instead of the id.

<a id="question-set-2"></a>

## <span style="color: #eab308">Question set 2</span>

> **Q: explain `const projects = useQuery(api.projects.get1, isAuthenticated ? {} : "skip");` and the `projects?.map(({ _id, ownerId, name }) => (...))` block that renders each project.**
>
> **`const projects = useQuery(api.projects.get1, isAuthenticated ? {} : "skip");`**
> - **`useQuery`** (from `convex/react`) doesn't work like a normal async function call — it registers a **live, standing subscription** to a Convex query. Convex re-runs `get1` on the server and pushes fresh results down a WebSocket automatically whenever the underlying data changes, no manual refetching or polling involved.
> - **`api.projects.get1`** is a typed reference (generated into `convex/_generated/api`) pointing at the `get1` query defined in `convex/projects.ts` — the one that checks `ctx.auth.getUserIdentity()` and returns only the current user's own projects.
> - **`isAuthenticated ? {} : "skip"`** is the second argument — the query's *arguments*, or the sentinel `"skip"`. `isAuthenticated` comes from `useConvexAuth()`. This is a ternary expression: while `isAuthenticated` is `false`, the value is `"skip"` — a special string `useQuery` understands as "don't run this query at all." `projects` stays `undefined` the entire time. Once Convex confirms a signed-in identity, `isAuthenticated` flips to `true`, the argument becomes `{}` (an empty args object, since `get1` takes no parameters), and `useQuery` starts the real subscription — `get1` runs, filtered to `ownerId === identity.subject`, and `projects` becomes the resulting array.
> - Any time that second-argument *value* changes (`"skip"` ↔ `{}`), Convex tears down the old subscription (if any) and starts fresh — this is what makes sign-in/sign-out correctly start/stop the live data feed.
>
> **`{projects?.map(({ _id, ownerId, name }) => ( ... ))}`**
> - **`projects?.`** — optional chaining. Since `projects` is `undefined` until the query above actually returns data (or is skipped entirely while signed out), this guards against calling `.map()` on `undefined`, which would throw. If `projects` is `undefined`, the whole expression evaluates to `undefined`, and React simply renders nothing there.
> - **`.map(({ _id, ownerId, name }) => ( ... ))`** — once `projects` is a real array, this transforms each project *document* into a piece of JSX. `({ _id, ownerId, name })` destructures those three fields directly out of each element as it's passed in (a Convex document also carries `_creationTime`, unused here). The arrow's body is wrapped in `( ... )` right after `=>` — the implicit-return form, so whatever JSX is inside is automatically that array element's produced value.
> - **`<div className="..." key={_id.toString()}>`** — one styled box per project. `key` is required by React whenever rendering an array of elements, so it can correctly track which box corresponds to which project across re-renders (added/removed/reordered items). `_id` is the right choice here since it's guaranteed unique per document — unlike `ownerId`, which would be identical across every project this particular query returns (they all belong to the same signed-in user).
> - **`{name} ({ownerId})`** — renders the project's name, followed by its owner's Clerk user ID in parentheses, as the box's visible text.
>
> **How the two connect:** `useQuery` is what keeps `projects` continuously up to date in the background (or `undefined` while signed out); `.map()` is what turns whatever `projects` currently holds into visible boxes on the page, re-running automatically every time React re-renders in response to that live data changing.

> **Q: would it make sense to create `providers.tsx` and put everything under `<Authenticated>`, as in the reference project's `polaris-main/src/components/providers.tsx`?**
>
> Yes, it would — that's a cleaner pattern than what's currently in `page.tsx`. Right now, auth-gating is done ad hoc per-query (`isAuthenticated ? {} : "skip"` plus manual empty-state handling); the reference's `providers.tsx` moves that decision to one place at the root, using Convex's `<Authenticated>`/`<Unauthenticated>`/`<AuthLoading>` components to show the real app, a sign-in prompt, or a loading spinner respectively — so individual pages/queries can just assume they're authenticated and skip the boilerplate entirely.
>
> The main tradeoff: this makes the **entire app** require sign-in to see anything at all (no page renders unless you're logged in) — which fits a private tool like the reference app, but is a real behavior change from `my_polaris` today, where the page is currently visible/usable-looking even when signed out (only the actual Convex calls are blocked). If signed-out visitors should ever see any public-facing content, `<Authenticated>` would need to wrap only part of the tree, not everything.

> **Q: why does `layout.tsx` not make sure that if anything in `{children}` is not authenticated, then it would not be shown?**
>
> Because nothing in `layout.tsx` currently tells it to gate `{children}` — the auth-conditional rendering that exists there (`<Show when="signed-out">` / `<Show when="signed-in">`) is only wrapped around the header's Sign In/Sign Up buttons vs. the `UserButton`. `{children}` right after the header is rendered **unconditionally** — there's no `<Show>`, no `<Authenticated>`, no check of any kind around it. It's simply never been written; not something the code is attempting and failing at, just absent.
>
> **Where auth enforcement actually happens instead, in this app: the data layer, not the UI layer.**
> - In `page.tsx`, `useQuery(api.projects.get1, isAuthenticated ? {} : "skip")` means the query is *skipped* when signed out — so `projects` stays `undefined` and nothing renders in that specific list.
> - In `convex/projects.ts`, both `create1` and `get1` call `ctx.auth.getUserIdentity()` and throw if there's no identity — so even if someone bypassed the UI, the actual database operations reject unauthenticated callers.
>
> So the **data** is protected, but the **UI shell** is not — a signed-out visitor still sees the full page structure: the "Project name" input, the "Create Project" button, the whole layout. They just can't successfully create anything (the mutation throws) or see any projects (the query never runs). It's a page that *looks* interactive but silently does nothing useful when you're not signed in, rather than a page that's replaced by a sign-in prompt.
>
> This is exactly the gap the reference project's `providers.tsx` pattern closes: wrapping `{children}` in Convex's `<Authenticated>` would stop that content from rendering *at all* when signed out, showing an `<Unauthenticated>` view (like a "please sign in" screen) instead — moving the enforcement from "the data quietly refuses to load" to "the UI itself refuses to appear."

<a id="new-plan"></a>

## <span style="color: #eab308">New Plan</span>

Goal: signed-out visitors see only a title and sign-in/sign-up controls (`WelcomeView`); signed-in users see exactly what the app already showed before (the project-name input, the "Create Project" button, and the live project list).

**1. `WelcomeView` was created as planned**, at `src/components/welcome-view.tsx`:
```tsx
"use client"

import {
  SignInButton,
  SignUpButton,
} from "@clerk/nextjs";

import { Button } from "@/components/ui/button";

export function WelcomeView() {
  return (
    <div className="flex min-h-screen flex-col items-center justify-center gap-6">
      <h1 className="text-3xl font-bold">Welcome to My_Polaris</h1>
      <div className="flex gap-3">
        <SignInButton>
          <Button variant="outline">Sign In</Button>
        </SignInButton>
        <SignUpButton>
          <Button>Sign Up</Button>
        </SignUpButton>
      </div>
    </div>
  );
}
```

**2. The first attempt followed the original plan literally** — `<Authenticated>` / `<Unauthenticated>` were added directly into `layout.tsx`'s own JSX, wrapping `{children}`. This broke immediately with a runtime error:
```
TypeError: createContext only works in Client Components. Add the "use client" directive at the top of the file to use it.
```
**Why:** `layout.tsx` has no `"use client"` directive, so Next.js's App Router treats it as a **Server Component**. `Authenticated`/`Unauthenticated` come from `convex/react`, which calls React's `createContext` at module-evaluation time (the moment it's imported, before any component even renders) to set up the context those components read from — and `createContext` isn't available in the server-components module graph. So simply importing `convex/react` directly into `layout.tsx` failed, regardless of how it was used afterward.
Adding `"use client"` to `layout.tsx` itself was considered and rejected: `layout.tsx` exports `metadata`, and Next.js explicitly disallows exporting `metadata` from a file marked `"use client"` — that would have traded one error for another.

**3. The actual fix: move `<Authenticated>` / `<Unauthenticated>` into `ConvexClientProvider.tsx` instead**, since that file already had `"use client"` at its top — it was already the app's existing client boundary for anything Convex/Clerk-context-related. `ConvexClientProvider.tsx` now reads:
```tsx
"use client";

import { ReactNode } from "react";
import { Authenticated, Unauthenticated, ConvexReactClient } from "convex/react";
import { ConvexProviderWithClerk } from "convex/react-clerk";
import { useAuth } from "@clerk/nextjs";
import { WelcomeView } from "@/components/welcome-view";

const convex = new ConvexReactClient(process.env.NEXT_PUBLIC_CONVEX_URL!);

export function ConvexClientProvider({ children }: { children: ReactNode }) {
  return (
    <ConvexProviderWithClerk client={convex} useAuth={useAuth}>
      <Authenticated>
        {children}
      </Authenticated>
      <Unauthenticated>
        <WelcomeView />
      </Unauthenticated>
    </ConvexProviderWithClerk>
  );
}
```
`layout.tsx` went back to a much simpler form — no `Authenticated`/`Unauthenticated` imports, no auth-conditional JSX of its own:
```tsx
<ConvexClientProvider>
  {children}
</ConvexClientProvider>
```
It stays a Server Component, `metadata` keeps working, and `convex/react`'s context-creating exports are only ever imported inside a file actually marked `"use client"`.

**4. A dedicated `AppHeader` was added, rendered inside `<Authenticated>`**, since the app is expected to grow beyond this one page and a header is something every future authenticated page will want — keeping it in `ConvexClientProvider.tsx`, right alongside `{children}`, means it's guaranteed to render once (and only when) a user is signed in, without every individual page needing to remember to include it. New file, `src/components/app-header.tsx`:
```tsx
"use client";

import { UserButton } from "@clerk/nextjs";

export function AppHeader() {
  return (
    <header className="flex items-center justify-end gap-4 p-4">
      <UserButton />
    </header>
  );
}
```
`UserButton` (Clerk's avatar menu) already includes a "Sign out" action in its dropdown, so this single component covers the "sign-out button" requirement without needing a separate explicit button. `ConvexClientProvider.tsx`'s `<Authenticated>` branch now reads:
```tsx
<Authenticated>
  <AppHeader />
  {children}
</Authenticated>
```
As part of this same cleanup, the two leftover issues from the previous attempt were also resolved: the unused `WelcomeView` import in `layout.tsx` is gone, and so are the now-dead `Show`/`SignInButton`/`SignUpButton`/`UserButton` imports from `@clerk/nextjs` that `layout.tsx` no longer uses — `layout.tsx` now only imports `ClerkProvider` from Clerk.

**Result:** signed out → `<Unauthenticated>` renders `WelcomeView` (title + sign-in/sign-up), and nothing else in the app is reachable or rendered. Signed in → `<Authenticated>` renders `AppHeader` (with a working sign-out control via `UserButton`) followed by `{children}`, i.e. the project-creation-and-list page — functionally unchanged from before, now with a visible, working way to see who's signed in and sign out.

<a id="ai-sdk-blocking"></a>

## <span style="color: #eab308">AI-SDK BLOCKING</span>

**I want to use the AI SDK with xAI. If I create a file at `src/app/api/demo/blocking/route.ts`, what's the point, and is it a Next.js pattern?**

Yes — this is the Next.js App Router's **Route Handlers** pattern. Any file at `src/app/api/<path>/route.ts` (that exact filename) becomes a server-side API endpoint reachable at the matching URL — `src/app/api/demo/blocking/route.ts` is reachable at `/api/demo/blocking`. Inside it, you export functions named after HTTP methods (`export async function GET(request) {...}`, or `POST`, etc.), and whatever you return becomes the HTTP response.

**Why a server route instead of calling xAI directly from a component:** an xAI API key is a secret. Calling the AI SDK's `generateText({ model: xai(...) })` directly inside a `"use client"` component would run that code in the browser, which means the key would have to ship inside client-side JavaScript — visible to anyone via dev tools. A Route Handler runs only on the server, so the key stays server-side (read from `process.env`, never sent to the browser); the client instead does a plain `fetch("/api/demo/blocking")`, and the route handler is the only thing that ever talks to xAI directly.

**On the filename — worth getting exactly right, since it tripped up twice:** Next.js recognizes only the exact filename **`route.ts`** in that folder position (note: *r-o-u-t-e*, singular, ending in "e" — not "wouts.ts", and not "routs.ts" either). A file with any other name in `src/app/api/...` is just an inert TypeScript file — it won't be wired to any URL at all, and Next.js won't error or warn you; it'll simply never be reached. Only the literal name `route.ts` triggers this behavior, the same way only `page.tsx` and `layout.tsx` are special-cased elsewhere in the App Router.

**Worth confirming:** naming the folder `blocking` strongly suggests a second one is planned — probably `src/app/api/demo/streaming/route.ts` — to compare the AI SDK's two response modes: `generateText` (waits for the full response before returning anything — "blocking") vs `streamText` (sends tokens back as they're generated).

> **Q: where do I put the xai key?**
>
> Put it in **`.env`**, following the same pattern this project already uses for Clerk/Convex secrets — **no `NEXT_PUBLIC_` prefix**, since it must stay server-side only:
> ```
> XAI_API_KEY=your-key-here
> ```
> `XAI_API_KEY` is the exact name to use — the AI SDK's `xai` provider (`@ai-sdk/xai`) automatically reads that specific environment variable by default, so it doesn't need to be passed explicitly:
> ```ts
> // src/app/api/demo/blocking/route.ts
> import { xai } from "@ai-sdk/xai";
> import { generateText } from "ai";
>
> export async function GET() {
>   const { text } = await generateText({
>     model: xai("grok-4.7"),
>     prompt: "...",
>   });
>   return Response.json({ text });
> }
> ```
> **Why `.env` and not `NEXT_PUBLIC_...`:** any env var prefixed `NEXT_PUBLIC_` gets bundled into client-side JavaScript and is visible in the browser — that's how `NEXT_PUBLIC_CONVEX_URL` and `NEXT_PUBLIC_CLERK_PUBLISHABLE_KEY` work in this project, because those specific values are meant to be public. An API key is the opposite case — it must never reach the browser, so it stays unprefixed, readable only via `process.env.XAI_API_KEY` inside server-only code like a Route Handler. `.env` is already `.gitignore`d in this repo, so it won't end up in git either.
>
> One setup step: restart `npm run dev` after adding it — env vars are only read when the dev server starts, same as with the Clerk/Convex keys earlier.

> **Q: why is `route.ts` a GET rather than a POST? Do I not POST the prompt?**
>
> Good catch — and the honest answer is: right now nothing is actually being sent to the server, which is exactly why GET "works" here.
>
> The prompt is **hardcoded inside the route handler itself**:
> ```ts
> export async function GET() {
>   const { text } = await generateText({
>     model: groq('openai/gpt-oss-20b'),
>     prompt: 'Write a vegetarian lasagna recipe for 4 people.',  // hardcoded
>   });
>   return Response.json({ text });
> }
> ```
> The client (`fetch("/api/demo/blocking")`) sends no body, no data — it's just asking the server to run its own pre-written prompt and hand back the result. That's a pure "give me something" request with no client-supplied input, which is exactly what GET is *for*.
>
> **If the user types their own prompt and submits it, that does need to become POST.** The distinction (standard REST/HTTP convention, and `fetch` itself enforces part of it):
> - **GET** — retrieve something; conventionally carries no request body. (`fetch` actually throws a `TypeError` if you try to attach a `body` to a GET request — it's disallowed by the Fetch spec, not just a style guideline.)
> - **POST** — submit data for the server to process; the natural place for a request body.
>
> So once there's a real prompt coming from the user, the route needs to change shape:
> ```ts
> // src/app/api/demo/blocking/route.ts
> export async function POST(request: Request) {
>   const { prompt } = await request.json();
>
>   const { text } = await generateText({
>     model: groq('openai/gpt-oss-20b'),
>     prompt,
>   });
>
>   return Response.json({ text });
> }
> ```
> And the page's fetch call would need to send that body:
> ```ts
> const res = await fetch("/api/demo/blocking", {
>   method: "POST",
>   headers: { "Content-Type": "application/json" },
>   body: JSON.stringify({ prompt: userInput }),
> });
> ```

**Updated: `route.ts` now accepts a POSTed prompt, and `page.tsx` lets the user type one**

Following the GET-vs-POST discussion above, both files were changed together. `src/app/api/demo/blocking/route.ts`:
```ts
import { groq } from '@ai-sdk/groq';
import { generateText } from 'ai';

export async function POST(request: Request) {
  const { prompt } = await request.json();

  const { text } = await generateText({
    model: groq('openai/gpt-oss-20b'),
    prompt,
  });

  return Response.json({ text });
}
```
`GET` became `POST`, and the hardcoded recipe prompt was replaced with `const { prompt } = await request.json();` — reading whatever prompt the client actually sends, instead of running a fixed string every time.

`src/app/demo/blocking/page.tsx`:
```tsx
"use client";

import { useState } from "react";
import { Button } from "@/components/ui/button";
import { Input } from "@/components/ui/input";

export default function DemoPage() {
  const [prompt, setPrompt] = useState("");
  const [text, setText] = useState("");
  const [loading, setLoading] = useState(false);

  const handleClick = async () => {
    setLoading(true);
    const res = await fetch("/api/demo/blocking", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ prompt }),
    });
    const data = await res.json();
    setText(data.text);
    setLoading(false);
  };

  return (
    <main className="flex min-h-screen flex-col items-center justify-center gap-6 p-8">
      <div className="flex w-full max-w-sm gap-2">
        <Input
          placeholder="Type a prompt"
          value={prompt}
          onChange={(e) => setPrompt(e.target.value)}
        />
        <Button onClick={handleClick} disabled={loading || !prompt.trim()}>
          {loading ? "Generating..." : "Generate"}
        </Button>
      </div>
      {text && <pre className="max-w-2xl whitespace-pre-wrap">{text}</pre>}
    </main>
  );
}
```

> **How `src/app/demo/blocking/page.tsx` works, in detail**
>
> **State — three independent pieces, via `useState`:**
> - `prompt` — the live text of whatever the user is typing, starting as `""`.
> - `text` — the generated response once it comes back, starting as `""`.
> - `loading` — a boolean tracking whether a request is currently in flight, starting `false`.
>
> **The input — a controlled component, same pattern as the homepage's "Project name" input:**
> ```tsx
> <Input
>   placeholder="Type a prompt"
>   value={prompt}
>   onChange={(e) => setPrompt(e.target.value)}
> />
> ```
> `value={prompt}` pins the box's displayed text to state; `onChange` fires on every keystroke and updates `prompt` to match, via `setPrompt(e.target.value)`. Without this pairing the box would appear frozen — see the earlier notebook cells on controlled inputs for the full explanation of why.
>
> **The button and its disabled state:**
> ```tsx
> <Button onClick={handleClick} disabled={loading || !prompt.trim()}>
>   {loading ? "Generating..." : "Generate"}
> </Button>
> ```
> `onClick={handleClick}` is a bare function reference — React calls it when the button is actually clicked, not during render. `disabled={loading || !prompt.trim()}` keeps the button unclickable in **two** situations: while a request is already running (`loading` is `true`, preventing duplicate submissions), or while the input is empty/whitespace-only (`!prompt.trim()`, same empty-guard pattern used on the homepage). The label itself swaps between `"Generate"` and `"Generating..."` based on `loading`, so there's visible feedback during the multi-second wait for the LLM call.
>
> **`handleClick` — the actual fetch flow:**
> ```tsx
> const handleClick = async () => {
>   setLoading(true);
>   const res = await fetch("/api/demo/blocking", {
>     method: "POST",
>     headers: { "Content-Type": "application/json" },
>     body: JSON.stringify({ prompt }),
>   });
>   const data = await res.json();
>   setText(data.text);
>   setLoading(false);
> };
> ```
> 1. `setLoading(true)` — immediately disables the button and flips its label, before the network call even starts.
> 2. `fetch("/api/demo/blocking", { method: "POST", ... })` — unlike a plain GET fetch, this explicitly sets the HTTP method to POST and attaches a JSON body. `headers: { "Content-Type": "application/json" }` tells the server how to interpret that body; `body: JSON.stringify({ prompt })` serializes the current `prompt` state into a JSON string — `{ prompt }` is shorthand for `{ prompt: prompt }`.
> 3. On the server, `route.ts`'s `POST` handler reads it back out with `await request.json()`, destructuring `prompt` from the parsed body, then passes that exact string into `generateText`.
> 4. `const data = await res.json();` parses the route's `Response.json({ text })` back into a JS object; `setText(data.text)` stores the generated response in state.
> 5. `setLoading(false)` — re-enables the button and restores its label, now that the request has resolved.
>
> **Rendering the result:**
> ```tsx
> {text && <pre className="max-w-2xl whitespace-pre-wrap">{text}</pre>}
> ```
> While `text` is still `""` (falsy), this expression evaluates to `""` and React renders nothing. Once `text` holds a real string (truthy), the `<pre>` block renders — `whitespace-pre-wrap` makes line breaks in the generated text actually show as line breaks instead of collapsing onto one line, which plain HTML text otherwise does.
>
> **What's still simple/not handled:** no error handling if the fetch fails or Groq errors out (same "no try/catch" tradeoff as `handleCreate` on the homepage), and pressing Enter in the input does nothing — only clicking the button triggers a request, unlike the homepage's input which also listens for the Enter key.

> **Q: how do we know how to write the header: `headers: { "Content-Type": "application/json" }`?**
>
> `Content-Type` is a standard HTTP header telling the receiver what format the body is in — `"application/json"` is the registered MIME type for a JSON string, which is what `JSON.stringify({ prompt })` produces.
>
> Worth knowing: `fetch`'s default `Content-Type` (if omitted) is `text/plain`, not `application/json`, even for a JSON string body — so setting it explicitly matters on stricter backends, even though Next.js's `request.json()` here actually ignores the header and just parses the raw body text regardless.
>
> `headers` itself is just a plain object of header-name → value pairs — add more the same way if a future endpoint needs them (e.g. `Authorization`).

<a id="ai-sdk-streaming"></a>

## <span style="color: #eab308">AI_SDK STREAMING</span>

**The streaming version of the blocking demo — same idea, `streamText` instead of `generateText`, plus the plumbing that difference requires.**

`src/app/api/demo/streaming/route.ts`:
```ts
import { streamText, createTextStreamResponse, toTextStream } from 'ai';
import { groq } from '@ai-sdk/groq';

export async function POST(request: Request) {
  const { prompt } = await request.json();

  const result = streamText({
    model: groq('openai/gpt-oss-20b'),
    prompt,
  });

  return createTextStreamResponse({ stream: toTextStream({ stream: result.stream }) });
}
```

`src/app/demo/streaming/page.tsx`:
```tsx
"use client";

import { useState } from "react";
import { Button } from "@/components/ui/button";
import { Input } from "@/components/ui/input";

export default function StreamingDemoPage() {
  const [prompt, setPrompt] = useState("");
  const [text, setText] = useState("");
  const [loading, setLoading] = useState(false);

  const handleClick = async () => {
    setLoading(true);
    setText("");
    const res = await fetch("/api/demo/streaming", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ prompt }),
    });

    const reader = res.body!.getReader();
    const decoder = new TextDecoder();
    let result = "";
    while (true) {
      const { done, value } = await reader.read();
      if (done) break;
      result += decoder.decode(value);
      setText(result);
    }

    setLoading(false);
  };

  return (
    <main className="flex min-h-screen flex-col items-center justify-center gap-6 p-8">
      <div className="flex w-full max-w-sm gap-2">
        <Input
          placeholder="Type a prompt"
          value={prompt}
          onChange={(e) => setPrompt(e.target.value)}
        />
        <Button onClick={handleClick} disabled={loading || !prompt.trim()}>
          {loading ? "Generating..." : "Generate"}
        </Button>
      </div>
      {text && <pre className="max-w-2xl whitespace-pre-wrap">{text}</pre>}
    </main>
  );
}
```

> **How the streaming version works, in detail — and how it differs from the blocking version**
>
> **`route.ts` — three changes from the blocking route:**
> 1. **`streamText` instead of `generateText`.** `generateText` is `await`-ed and only resolves once the entire response has been generated — `streamText` returns *immediately* with an in-progress stream object; the text keeps arriving from Groq in the background while your code has already moved on to the next line.
> 2. **No more `await` on the result, and no more `const { text }`.** `generateText`'s result is the finished text, ready to read. `streamText`'s result (`result`) is an object wrapping a live stream (`result.stream`) — there is no finished `text` value available yet at the point you return from the handler, by design.
> 3. **`return createTextStreamResponse({ stream: toTextStream({ stream: result.stream }) })` instead of `return Response.json({ text })`.** `Response.json(...)` sends one complete JSON body in a single response. A stream can't be described as one JSON value up front — `toTextStream` adapts the AI SDK's internal stream format into a plain text byte stream, and `createTextStreamResponse` wraps that into a `Response` whose body is that live stream itself. The HTTP response starts immediately and keeps sending bytes as Groq produces them, rather than waiting for everything first.
>
> **`page.tsx` — `res.json()` no longer works, because the body isn't one JSON object anymore:**
> ```tsx
> const reader = res.body!.getReader();
> const decoder = new TextDecoder();
> let result = "";
> while (true) {
>   const { done, value } = await reader.read();
>   if (done) break;
>   result += decoder.decode(value);
>   setText(result);
> }
> ```
> - **`const reader = res.body!.getReader();`** — `res.body` is typed `ReadableStream<Uint8Array> | null`; the `!` is TypeScript's **non-null assertion operator**, telling the compiler "trust me, this is never null here." It's compile-time-only and produces **zero runtime code** — it doesn't actually check anything. If `res.body` genuinely were `null`, `.getReader()` would still throw a real error at runtime; the `!` only silences TypeScript's static warning, it provides no actual safety. `.getReader()` itself returns a `ReadableStreamDefaultReader`, an object whose `.read()` method (called in the loop below) pulls one chunk of bytes at a time from the stream, on demand.
> - **`const decoder = new TextDecoder();`** — the stream delivers raw bytes (`Uint8Array` chunks), not strings; `TextDecoder` converts those bytes into readable text via its `.decode()` method.
> - **Where `getReader` and `TextDecoder` come from — not React, not Next.js:** both are **Web Platform APIs**, global objects built into the JavaScript runtime itself, not provided by either library. `ReadableStream`/`getReader()` comes from the **WHATWG Streams Standard**; `TextDecoder` comes from the **WHATWG Encoding Standard**. That's why neither has an `import` statement, unlike `useState` (from `"react"`) or `Button`/`Input` (from local files) — they're globals available automatically, the same category as `fetch`, `Response`, or `Promise`. Browsers have implemented both standards natively for years; Node.js has had them as globals since Node 18 (Streams) and Node 11 (`TextDecoder`). Next.js doesn't implement or wrap either — it just runs code inside environments (the browser, or Node.js/Edge Runtime on the server) that already have them built in.
> - **The `while (true)` loop** repeatedly calls `reader.read()`, which returns a promise resolving to `{ done, value }`. `value` is the next chunk of bytes (or `undefined` on the final call); `done` becomes `true` once the stream has no more data, at which point `break` exits the loop.
> - **`result += decoder.decode(value)`** appends each newly-decoded chunk onto the running total, and **`setText(result)`** updates state with that growing string on *every single chunk* — this is what makes the text visibly grow on screen as the model generates it, instead of appearing all at once at the end.
> - **`setText("")` at the start of `handleClick`** clears any previous result before a new request starts, so a second generation doesn't briefly show old text glued to the front of the new stream.
>
> **The core behavioral difference, side by side:**
> | | Blocking (`/demo/blocking`) | Streaming (`/demo/streaming`) |
> |---|---|---|
> | AI SDK function | `generateText` | `streamText` |
> | Route response | `Response.json({ text })` — one complete value | `createTextStreamResponse(...)` — a live byte stream |
> | Client reads with | `await res.json()` | `res.body!.getReader()` + a read loop |
> | `setText` calls | once, with the full result | repeatedly, once per chunk, building up the text |
> | User sees | nothing, then the full recipe all at once | words appearing progressively as they're generated |
>
> Everything else — the `Input`/`Button` structure, the `disabled={loading || !prompt.trim()}` guard, the `{text && <pre>...}` render — is identical to the blocking page on purpose, so the only real difference on screen is *how* the text arrives, not the surrounding UI.

<a id="inngest"></a>

## <span style="color: #eab308">INNGEST</span>

**What Inngest is for**

Inngest is an event-driven **durable execution** platform — it lets you write background jobs, scheduled tasks, and multi-step workflows as ordinary TypeScript functions, while Inngest itself handles the infrastructure that's normally painful to build yourself: queueing, retries on failure, concurrency/rate limiting, scheduling (cron), and observability (seeing what ran, what failed, and why).

**Why this is a different thing from a Route Handler.** A Route Handler like `route.ts` in this project runs once, synchronously, in direct response to an HTTP request, and must finish within that request's lifetime. Inngest functions instead run as background jobs — triggered by an **event** (a named payload, sent by your code) or a **schedule**, not directly by an HTTP request — and can run far longer, be broken into retryable **steps**, pause and resume, and survive failures by automatically retrying just the failed step rather than the whole job from scratch.

**Core concepts, briefly:**
- **Events** — named payloads that trigger functions (e.g. `"user.signed.up"`, or any custom event name you define and send yourself). One event can trigger multiple functions.
- **Functions** — the actual workflow code, split into one or more **steps** (`step.run(...)`). Each step's result is durably saved, so if a later step fails and the function retries, steps that already succeeded aren't re-run.
- **Durable execution** — the defining feature: a function keeps running until it truly completes, surviving crashes and restarts, with Inngest managing retries and state on your behalf.
- **Cron / scheduled functions** — functions can also trigger on a schedule instead of an event, covering "run this every night" style jobs.

**How it plugs into Next.js:** a single Route Handler, conventionally `src/app/api/inngest/route.ts`, acts as the "serve" endpoint. It doesn't run your job logic itself — it registers your Inngest functions and lets Inngest's own servers call back into your app to actually execute them, on their schedule, not the browser's.

**Why it showed up in the reference project, and whether this project needs it.** The reference `polaris-main.zip` project used Inngest for things like GitHub import/export jobs — work that's too slow or unreliable to run synchronously inside one request-response cycle, and that benefits from automatic retries. For the AI SDK demo routes built so far in `my_polaris` (`/api/demo/blocking`, `/api/demo/streaming`), **Inngest isn't needed** — both are quick, single-shot calls that finish comfortably within a normal request, which is exactly why "do I need Inngest for streaming" (an earlier question in this notebook) was answered no. It would become relevant if this project later added something like a long-running or multi-step AI workflow — e.g. "import a repo, then generate several AI-written summaries, retrying any that fail" — where reliability and background processing start to matter more than an instant response.

> **Q: assuming this app will use Inngest, what are all the commands to start everything, in order?**
>
> Three terminals, each left running:
>
> **Terminal A — Convex (leave running):**
> ```powershell
> npx convex dev
> ```
>
> **Terminal B — Next.js (leave running):**
> ```powershell
> npm run dev
> ```
>
> **Terminal C — Inngest Dev Server (leave running):**
> ```powershell
> inngest dev
> ```
> If `inngest` isn't recognized (PATH not set up yet), use the full path instead:
> ```powershell
> & "$env:LOCALAPPDATA\inngest\inngest.exe" dev
> ```
>
> The order between these three doesn't matter functionally — they're independent processes that only need to *all* be running at the same time, not started in a specific sequence. Once all three are up, open **http://localhost:3000** for the app itself and **http://localhost:8288** for the Inngest dashboard.
>
> **One thing still missing before Inngest would actually show anything connected:** the dev server auto-discovers apps via an `/api/inngest` endpoint, which `my_polaris` doesn't have yet (see above — nothing in this project uses Inngest currently). Running all three servers today would still show the Inngest dashboard with no connected app, exactly as it did when just testing the CLI. Wiring up an actual Inngest client and at least one function under `src/app/api/inngest/route.ts` is a separate, not-yet-done step for whenever this app genuinely needs background jobs.

> **Q: I added `client.ts` and `route.ts` in 2 inngest directories. What does this do, and why this file structure?**
>
> Two files, two different jobs.
>
> **`src/inngest/client.ts`** — defines the Inngest **client**, the single object representing this app's identity to Inngest (`id: "my_polaris"`). This is what you'd use both to *send* events (`inngest.send(...)`) and to *define* functions (`inngest.createFunction(...)`) anywhere else in the app.
> ```ts
> import { Inngest } from "inngest";
> export const inngest = new Inngest({ id: "my_polaris" });
> ```
>
> **`src/app/api/inngest/route.ts`** — the "serve" endpoint, using the same Route Handler pattern as the AI SDK demo routes (reserved filename `route.ts`, mapped to `/api/inngest`). `serve()` (from `inngest/next`, the Next.js-specific adapter) takes the client plus a `functions` array and produces the actual `GET`/`POST`/`PUT` handlers Next.js needs:
> ```ts
> import { serve } from "inngest/next";
> import { inngest } from "../../../inngest/client";
>
> export const { GET, POST, PUT } = serve({
>   client: inngest,
>   functions: [],
> });
> ```
> This is exactly the endpoint the dev server's "Scanning for available serve handlers" message (from earlier) was looking for — now that it exists, the dashboard should show `my_polaris` connected. `functions: []` is still empty, so there's nothing to actually *run* yet — this is the scaffold, wired but with no jobs registered.
>
> **Why two files instead of one:** `client.ts` holds identity/config; `route.ts` holds "how this app exposes itself to Inngest over HTTP." The split matters practically — a `route.ts` file is only allowed to export HTTP method handlers (plus a few specific named exports), not arbitrary values, so the client *can't* live inside it if anything else in the app ever needs to import that same client to send an event (e.g. from a Convex mutation, or another route). Keeping it in its own module makes it importable from anywhere.
>
> One small aside, not a bug: this project already uses the `@/` import alias elsewhere (`@/components/ui/button`, etc.) — `route.ts` could import via `@/inngest/client` instead of the relative `../../../inngest/client`, for consistency, though both resolve to the same file.

> **Q: why `PUT` in this log line: `PUT /api/inngest 200 in 19ms`?**
>
> `PUT` is the **registration/sync** call — it's how the Inngest Dev Server tells your app's endpoint "give me your current function list so I can register them."
>
> `serve()` in `route.ts` exports three handlers (`GET`, `POST`, `PUT`), each with a distinct job in Inngest's protocol:
> - **`GET`** — returns metadata about the app (its functions, SDK version) for introspection, and in dev mode also serves a small landing page. This is what the earlier "Scanning for available serve handlers" discovery step used to find the app at `/api/inngest` in the first place.
> - **`PUT`** — triggers the SDK to **register all functions with Inngest**. This is the sync step: the app reports "here's my current function list and configuration," and Inngest (dev server locally, or the cloud in production) records it.
> - **`POST`** — actually **executes** a function when triggered by a real event or schedule; the "run my code" call, with the request body carrying execution state.
>
> So `PUT /api/inngest 200` is the Dev Server registering/syncing with the app — it happens automatically once the Dev Server discovers the endpoint (via the earlier `GET` scan) or whenever the function list changes and needs re-syncing. It's a normal, expected part of the connection process, and the `200` means the sync succeeded.

>
> **A follow-up worth clarifying — who exports to whom.** Saying "`serve()` exports three handlers" conflates two different things:
> - **`route.ts` exports to Next.js.** `export const { GET, POST, PUT } = ...` uses the real JS/TS `export` keyword, making those three names available to whatever imports this module — which is **Next.js's own routing internals**. When Next.js builds/serves the app, it loads every `route.ts` file and looks specifically for exports named after HTTP methods to decide which verbs that URL responds to.
> - **`serve()` doesn't export anything — it just returns a plain object.** `serve()` is an ordinary function (from `inngest/next`) that internally builds and *returns* an object shaped like `{ GET: fn1, POST: fn2, PUT: fn3 }`. `const { GET, POST, PUT } = serve(...)` is plain **destructuring** of that returned object — no `export` keyword involved at that step.
>
> So the accurate chain is: `serve()` *returns* an object with three handler functions → that object gets destructured into three local variables → **those** get *exported* by `route.ts`, for Next.js to pick up. `serve()` itself has no involvement with the module `export` system — that only happens one line later, in `route.ts`.

<a id="inngest-diagram"></a>

## <span style="color: #eab308">INNGEST Diagram</span>

![Same Call, Two Paths — the identical generateText() call reached directly (blocking) versus through Inngest (background)](same-call-two-paths.svg)

> **Q: is the prompt first sent to Inngest, and does Inngest send it to Groq?**
>
> No — Inngest never talks to Groq at all. Inngest only ever talks to *your own app*. The actual sequence:
> 1. Browser → your route (`/api/demo/inngest`) → `inngest.send(...)`. This just delivers an event to Inngest's server — "here's something that happened," nothing more. Inngest stores it and looks up which function is subscribed to that event name.
> 2. Inngest then calls **back into your own app** — `POST /api/inngest` — saying "run this function now, here's the event data."
> 3. That `POST` lands in **your own code**, running inside your own Next.js app (the same codebase, just invoked by Inngest instead of by a browser). *That* code is what actually calls `generateText()`, which is what reaches Groq.
>
> So the "your Inngest function" box in the diagram represents code living back inside `my_polaris` — Inngest calling into it, not Inngest running it on its own infrastructure.

> **Q: why does the `inngest.send(...)` box have a dashed line back to the browser?**
>
> That line isn't a separate response from Inngest or from the `send` call itself — it's the response to the **browser's original request**. When the browser's `fetch` hits the route (`/api/demo/inngest`), that route handler does roughly:
> ```ts
> await inngest.send({ name: "recipe.requested", data: { prompt } });
> return Response.json({ status: "queued" });
> ```
> `inngest.send(...)` is fast — it's just handing an event to Inngest's server, not waiting for anything to actually run. As soon as that `await` resolves, the *same* route handler returns its own response, completing the *same* original HTTP request the browser made. It's drawn next to the `send` box because that's the last thing that happens before the route returns — but it's the same request/response pair the browser started, not a second round trip.

> **Q: after Groq generates the text, where does it get back to the browser?**
>
> It doesn't, in this diagram — deliberately. That's the actual cost of going async: the browser already got its "queued" reply *before* Groq even ran, so there's no live connection left to push the eventual result down. Inngest saves the result as durable step state on its own side, but getting it in front of the user needs a mechanism this project doesn't have yet — polling, a WebSocket, a webhook.
>
> The one that would fit naturally here, given what's already in `my_polaris`: the Inngest function could write its result into a Convex table once `generateText()` finishes, and the page would pick it up automatically via the same reactive `useQuery` pattern already used for the projects list — no new plumbing concept needed, just reusing what Convex already gives this app for free.

**Q: where do I write Inngest functions?**

Functions can live anywhere, but the natural fit given `src/inngest/client.ts` already exists is a sibling folder: **`src/inngest/functions/`**, one file per function.

**1. The function** — `src/inngest/functions/generate-text.ts`:
```ts
import { inngest } from "@/inngest/client";
import { groq } from "@ai-sdk/groq";
import { generateText } from "ai";

export const generateTextJob = inngest.createFunction(
  { id: "generate-text", triggers: { event: "text.requested" } },
  async ({ event, step }) => {
    const { text } = await step.run("generate", async () => {
      return generateText({
        model: groq("openai/gpt-oss-20b"),
        prompt: event.data.prompt,
      });
    });

    return text;
  }
);
```
Named `generateTextJob`, not plain `generateText` — that name is already taken in the same file by the AI SDK's own imported `generateText` function, so reusing it would collide. `step.run("generate", ...)` is what makes this a durable, retryable checkpoint — exactly what the diagram above showed.

This was checked against the actually-installed `inngest` package (v4.21.0)'s own type definitions rather than assumed from memory: `createFunction` takes two arguments, with `triggers` nested inside the first (`{ id, triggers: { event } }`, handler) — not an older three-argument `(config, trigger, handler)` shape some docs/examples show for other versions.

**2. Registered in `route.ts`**, replacing the empty `functions: []`:
```ts
import { serve } from "inngest/next";
import { inngest } from "@/inngest/client";
import { generateTextJob } from "@/inngest/functions/generate-text";

export const { GET, POST, PUT } = serve({
  client: inngest,
  functions: [generateTextJob],
});
```

**3. A trigger route** — `src/app/api/demo/inngest/route.ts`, mirroring the shape of the blocking/streaming demo routes, but instead of calling `generateText` itself, it hands the prompt to Inngest:
```ts
import { inngest } from "@/inngest/client";

export async function POST(request: Request) {
  const { prompt } = await request.json();

  const { ids } = await inngest.send({
    name: "text.requested",
    data: { prompt },
  });

  return Response.json({ status: "queued", eventId: ids[0] });
}
```
`inngest.send(...)` resolves with `{ ids: [...] }` — the event's own ID(s), confirmed against this project's actual `inngest` types (`SendEventBaseOutput`). This route returns almost immediately, since it's only waiting on the event being queued, not on `generateText` finishing.

**4. A page** — `src/app/demo/inngest/page.tsx`, using the same `Input` + `Button` pattern as the blocking demo:
```tsx
"use client";

import { useState } from "react";
import { Button } from "@/components/ui/button";
import { Input } from "@/components/ui/input";

export default function InngestDemoPage() {
  const [prompt, setPrompt] = useState("");
  const [status, setStatus] = useState("");
  const [loading, setLoading] = useState(false);

  const handleClick = async () => {
    setLoading(true);
    const res = await fetch("/api/demo/inngest", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ prompt }),
    });
    const data = await res.json();
    setStatus(`Queued — event ${data.eventId}. Check the Inngest dashboard (localhost:8288) to watch it run.`);
    setLoading(false);
  };

  return (
    <main className="flex min-h-screen flex-col items-center justify-center gap-6 p-8">
      <div className="flex w-full max-w-sm gap-2">
        <Input
          placeholder="Type a prompt"
          value={prompt}
          onChange={(e) => setPrompt(e.target.value)}
        />
        <Button onClick={handleClick} disabled={loading || !prompt.trim()}>
          {loading ? "Queuing..." : "Generate"}
        </Button>
      </div>
      {status && <p className="max-w-2xl text-sm text-muted-foreground">{status}</p>}
    </main>
  );
}
```

**Why this doesn't show the generated text inline, unlike the blocking/streaming demos.** This isn't a shortcut — it's the direct consequence of everything the diagram above already established: the browser's request returns as soon as the event is queued, before `generateText` has even started, so there's no open connection left for the eventual result to travel back on. Displaying it inline would require Inngest's separate **Realtime** feature (publishing from inside the step, subscribing from the browser with a token) — real, documented functionality, but a distinct chunk of infrastructure well beyond this flow, and not something to bolt on speculatively without verifying it end-to-end. For now, the page confirms the event was queued and points at the Inngest dashboard, where the run — and its generated text — is actually visible.

**Verified**: `npx tsc --noEmit` passes cleanly across the whole project with all of this in place.

<a id="start-all-terminals"></a>

## <span style="color: #eab308">Start-All_Terminals</span>

Running this app fully requires three long-running dev servers at once — Convex, Next.js, and Inngest — each normally started by typing a separate command in its own terminal. Rather than doing that by hand every time, a VS Code **task** now starts all three together, each in its own terminal panel.

**`.vscode/tasks.json`** defines four tasks: one per server, plus a compound task that runs all three in parallel.
```json
{
  "version": "2.0.0",
  "tasks": [
    {
      "label": "Convex Dev",
      "type": "shell",
      "command": "npx convex dev",
      "isBackground": true,
      "problemMatcher": [],
      "presentation": { "reveal": "always", "panel": "new", "group": "devServers" }
    },
    {
      "label": "Next Dev",
      "type": "shell",
      "command": "npm run dev",
      "isBackground": true,
      "problemMatcher": [],
      "presentation": { "reveal": "always", "panel": "new", "group": "devServers" }
    },
    {
      "label": "Inngest Dev",
      "type": "shell",
      "command": "& \"$env:LOCALAPPDATA\\inngest\\inngest.exe\" dev",
      "isBackground": true,
      "problemMatcher": [],
      "presentation": { "reveal": "always", "panel": "new", "group": "devServers" }
    },
    {
      "label": "Start All Dev Servers",
      "dependsOn": ["Convex Dev", "Next Dev", "Inngest Dev"],
      "dependsOrder": "parallel",
      "problemMatcher": []
    }
  ]
}
```

**Why each piece is there:**
- `"isBackground": true` — tells VS Code these commands never exit on their own (they're dev servers, not one-shot builds), so it shouldn't wait for them to "finish."
- `"presentation": { "panel": "new" }` — forces each task into its **own** terminal panel instead of sharing one, so the three servers' logs stay visually separate rather than interleaving.
- `"problemMatcher": []` — these commands aren't compilers with structured error output VS Code can parse, so this disables that matching rather than leaving it to guess.
- The `Inngest Dev` task uses the **full path** to `inngest.exe` (`$env:LOCALAPPDATA\inngest\inngest.exe`) rather than the bare `inngest` command, since PATH wasn't confirmed to resolve it globally yet at the time this was set up. If that's fixed later, the full path still works fine either way.
- The last task, `Start All Dev Servers`, has no command of its own — `"dependsOn"` lists the three server tasks, and `"dependsOrder": "parallel"` runs them all at once instead of one after another.

**To run it:**
1. `Ctrl+Shift+P` → **Tasks: Run Task**.
2. Select **Start All Dev Servers**.
3. Three new terminal panels open, one per server — switch between their tabs at the bottom of VS Code to watch each one.

**To stop everything:** close those three terminal panels, or click into each and press `Ctrl+C`.

> **Q: how to start `page.tsx` under `demo/inngest`?**
>
> `http://localhost:3000/demo/inngest`
>
> Since it's at `src/app/demo/inngest/page.tsx`, the App Router maps that folder path directly to `/demo/inngest` — same pattern as `/demo/blocking` and `/demo/streaming`.
>
> To actually see it work, all three dev servers need to be running (Next.js, Convex, and Inngest — use the **Start All Dev Servers** task above if they aren't already), and you need to be signed in, since every page in this app renders only while authenticated.
>
> After typing a prompt and clicking Generate, it won't show the result inline (covered in the INNGEST Diagram section) — check the Inngest dashboard at `http://localhost:8288` to actually watch the run happen and see the generated text.